# Amazon fine food reviews - Sentiment analysis

In [55]:
#importing the dependencies
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit,train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import TfidfVectorizer,ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix,f1_score


In [56]:
df=pd.read_csv("Reviews.csv")

In [57]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [58]:
#checking for null values in the dataset
df.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

In [59]:
#dropping the features that are not necessary
df=df.drop(columns=['Id','UserId','ProfileName','HelpfulnessNumerator','HelpfulnessDenominator','Time'])

In [60]:
#checking duplicate values in the Text and Summary features
df[['Text','Summary']].duplicated().sum()

np.int64(173484)

In [61]:
#only the first occurence of the rows are kept
df=df.drop_duplicates(subset=['Text','Summary'],keep='first')  

In [62]:
df.head()

,ProductId,Score,Summary,Text
0,B001E4KFG0,5,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,B00813GRG4,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,B000LQOCH0,4,"""Delight"" says it all",This is a confection that has been around a fe...
3,B000UA0QIQ,2,Cough Medicine,If you are looking for the secret ingredient i...
4,B006K2ZZ7K,5,Great taffy,Great taffy at a great price. There was a wid...


In [63]:
df.shape

(394970, 4)

### Mapping scores to sentiment
- 1-2 = 0
-    3      = 1
- 4-5 = 2


In [64]:
def map_sentiment(score):
    if(score<=2):
        return 0
    elif score==3:
        return 1
    else:
        return 2
df['Sentiment']=df['Score'].apply(map_sentiment)

In [65]:
y=df['Sentiment']

In [66]:
y

0         2
1         0
2         2
3         0
4         2
         ..
568449    2
568450    0
568451    2
568452    2
568453    2
Name: Sentiment, Length: 394970, dtype: int64

In [67]:
y.value_counts()

Sentiment
2    307787
0     57346
1     29837
Name: count, dtype: int64

In [84]:
x=df.drop(columns=['Score','Sentiment','ProductId'])

In [85]:
x.isnull().sum()

Summary    3
Text       0
dtype: int64

In [86]:
x['Summary']=x['Summary'].fillna('')

In [87]:
x.isnull().sum()

Summary    0
Text       0
dtype: int64

In [92]:
x['combined_text']=x['Summary']+x['Text']

In [93]:
#removing the negation words from the stop words
negation_words={'not', 'no', 'nor', 'never', 'none', 'nothing', 'nowhere', 
                   'neither', "n't", 'cannot', "don't", "doesn't", "didn't",
                   "won't", "wouldn't", "can't", "couldn't", "shouldn't",
                   "isn't", "aren't", "wasn't", "weren't", "hasn't", "haven't"}
#remvoing the domain words
domain_words={'amazon','product','item','br','href'}
custom_words=ENGLISH_STOP_WORDS-(negation_words)| domain_words

In [96]:
#GroupShuffleSplit is used to prevent reviews of same product end up inboth train and test data, which may cause data leakage
groups=df['ProductId']
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
train_idx,test_idx=next(gss.split(x,y,groups=groups))
x_train,x_test,y_train,y_test=x['combined_text'].iloc[train_idx],x['combined_text'].iloc[test_idx],y.iloc[train_idx],y.iloc[test_idx]


In [97]:
feature_extraction=TfidfVectorizer(min_df=3,stop_words=list(custom_words),lowercase=True, ngram_range=(1,2))
x_train_tfidf=feature_extraction.fit_transform(x_train)
x_test_tfidf=feature_extraction.transform(x_test)